# MERRA-2 Nighttime Recovery Statistics Processing

This notebook processes hourly MERRA-2 temperature data to compute daily nighttime recovery statistics.

**Nighttime window:** 8pm-6am UTC (hours 20-23, 0-5)  
**Temporal resolution:** Daily  
**Spatial resolution:** Grid cell level (lat, lon preserved)  
**Spatial extent:** US states only (masked using region_mask.nc)  
**Time period:** 1984-2025

## Metrics Computed Per Day

For each day, we compute nighttime hour counts for temperature thresholds:

### Threshold Metrics
1. `hours_below_18_above_0` - Strong recovery hours (0°C < T < 18°C)
2. `hours_below_21_above_0` - Reasonable recovery hours (0°C < T < 21°C)
3. `hours_below_24_above_0` - Weak recovery hours (0°C < T < 24°C)
4. `hours_above_21` - Poor recovery nights (T > 21°C)
5. `hours_above_24` - Very poor recovery nights (T > 24°C)
6. `hours_below_0` - Freezing conditions (T < 0°C)
7. `hours_below_neg5` - Very cold conditions (T < -5°C)
8. `hours_below_neg10` - Extremely cold conditions (T < -10°C)

## Output

- **Directory:** `processed_nighttime_recovery/`
- **File pattern:** `nighttime_recovery_{YYYYMM}.nc`
- **Format:** NetCDF4 with dimensions (time, lat, lon)
- **Variables:** 8 metrics (daily hour counts)
- **Fill value:** -1 for non-US areas and missing data

In [1]:
# Standard imports
import xarray as xr
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for imports
sys.path.insert(0, str(Path('../../').resolve()))

# Import configuration and src functions
import config
from src.processing import compute_daily_nighttime_stats, process_month

In [2]:
# Configuration from config.py
INPUT_DIR = config.DAILY_DATA_DIR
OUTPUT_DIR = config.PROCESSED_NIGHTTIME_DIR
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Time windows from config
NIGHTTIME_HOURS = config.NIGHTTIME_HOURS

# Date range
START_YEAR = 1984
END_YEAR = 2025

print(f"Input directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

# Metric descriptions for process_month
METRIC_NAMES = [
    'hours_below_18_above_0',
    'hours_below_21_above_0',
    'hours_below_24_above_0',
    'hours_above_21',
    'hours_above_24',
    'hours_below_0',
    'hours_below_neg5',
    'hours_below_neg10',
]

METRIC_DESCRIPTIONS = {
    'hours_below_18_above_0': ('Strong recovery hours (0°C < T < 18°C)',
                               'Daily count of nighttime hours with temperature between 0°C and 18°C'),
    'hours_below_21_above_0': ('Reasonable recovery hours (0°C < T < 21°C)',
                               'Daily count of nighttime hours with temperature between 0°C and 21°C'),
    'hours_below_24_above_0': ('Weak recovery hours (0°C < T < 24°C)',
                               'Daily count of nighttime hours with temperature between 0°C and 24°C'),
    'hours_above_21': ('Poor recovery nights (T > 21°C)',
                      'Daily count of nighttime hours with temperature above 21°C'),
    'hours_above_24': ('Very poor recovery nights (T > 24°C)',
                      'Daily count of nighttime hours with temperature above 24°C'),
    'hours_below_0': ('Freezing conditions (T < 0°C)',
                     'Daily count of nighttime hours with temperature below 0°C'),
    'hours_below_neg5': ('Very cold conditions (T < -5°C)',
                        'Daily count of nighttime hours with temperature below -5°C'),
    'hours_below_neg10': ('Extremely cold conditions (T < -10°C)',
                         'Daily count of nighttime hours with temperature below -10°C'),
}

Input directory: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/data/daily_data
Output directory: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/data/processed_nighttime_recovery


## Processing Functions

The processing functions are imported from `src/processing.py`:
- `compute_daily_nighttime_stats()`: Compute nighttime recovery statistics for a single day
- `process_month()`: Process all days in a month and save to NetCDF

See [src/processing.py](../../src/processing.py) for implementation details.

In [3]:
# Create wrapper function for nighttime processing
def process_nighttime_month(year, month):
    """
    Wrapper function to process nighttime recovery statistics for a month.
    Calls the generic process_month() with nighttime-specific parameters.
    """
    return process_month(
        year=year,
        month=month,
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        compute_stats_fn=lambda date, input_dir: compute_daily_nighttime_stats(date, input_dir, NIGHTTIME_HOURS),
        metric_names=METRIC_NAMES,
        metric_descriptions=METRIC_DESCRIPTIONS,
        output_prefix='nighttime_recovery',
        title='MERRA-2 Nighttime Recovery Statistics',
        description='Daily nighttime (8pm-6am UTC) temperature threshold hour counts',
        time_window_str='20-23, 0-5 (UTC)',
        mask_path=config.MASK_FILE,
        use_int16=True,
        fill_value=-1,
        skip_existing=True
    )

## Test Single Month

Test processing for a single month to verify the workflow.

In [4]:
# Test with a recent month
process_nighttime_month(2023, 6)

Output file nighttime_recovery_202306.nc already exists. Skipping.


True

## Process All Months (1984-2025)

Process all months in the dataset. This will skip any months that have already been processed.

In [5]:
# Process all years and months
for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        try:
            process_nighttime_month(year, month)
        except Exception as e:
            print(f"Error processing {year}-{month:02d}: {e}")
            continue

Output file nighttime_recovery_198401.nc already exists. Skipping.
Output file nighttime_recovery_198402.nc already exists. Skipping.
Output file nighttime_recovery_198403.nc already exists. Skipping.
Output file nighttime_recovery_198404.nc already exists. Skipping.
Output file nighttime_recovery_198405.nc already exists. Skipping.
Output file nighttime_recovery_198406.nc already exists. Skipping.
Output file nighttime_recovery_198407.nc already exists. Skipping.
Output file nighttime_recovery_198408.nc already exists. Skipping.
Output file nighttime_recovery_198409.nc already exists. Skipping.
Output file nighttime_recovery_198410.nc already exists. Skipping.
Output file nighttime_recovery_198411.nc already exists. Skipping.
Output file nighttime_recovery_198412.nc already exists. Skipping.
Output file nighttime_recovery_198501.nc already exists. Skipping.
Output file nighttime_recovery_198502.nc already exists. Skipping.
Output file nighttime_recovery_198503.nc already exists. Skipp

## Verify Output

Quick verification of the output files.

In [6]:
# List all output files
output_files = sorted(OUTPUT_DIR.glob('*.nc'))
print(f"Total output files: {len(output_files)}\n")

if output_files:
    # Inspect most recent file
    latest_file = output_files[-1]
    print(f"Inspecting: {latest_file.name}")
    ds = xr.open_dataset(latest_file)
    print(ds)
    print("\nData variables:")
    print(list(ds.data_vars))
    ds.close()

Total output files: 504

Inspecting: nighttime_recovery_202512.nc
<xarray.Dataset> Size: 10MB
Dimensions:                 (time: 31, lat: 51, lon: 95)
Coordinates:
  * time                    (time) datetime64[ns] 248B 2025-12-01 ... 2025-12-31
  * lat                     (lat) float64 408B 24.0 24.5 25.0 ... 48.0 48.5 49.0
  * lon                     (lon) float64 760B -125.0 -124.4 ... -66.88 -66.25
Data variables:
    hours_below_18_above_0  (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_21_above_0  (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_24_above_0  (time, lat, lon) timedelta64[ns] 1MB ...
    hours_above_21          (time, lat, lon) timedelta64[ns] 1MB ...
    hours_above_24          (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_0           (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_neg5        (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_neg10       (time, lat, lon) timedelta64[ns] 1MB ...
Attributes:
    title:     